# Section 5.7 — Sensitivity to Regulatory Constraints

Reproduces Table 11 from the thesis.  
Runs **SP (B-PHA)** on the **60-cage** fleet across three regional MAB limits:

| Scenario | Regional MAB |
|----------|--------------|
| Relaxed  | 35,000 t (base — non-binding) |
| Tangential | 24,000 t (just below sum of individual location MABs = 24,500 t) |
| Tight    | 18,000 t (binding) |

Individual location MABs are held fixed at their real values:
Loc1=4,000 t, Loc2=5,600 t, Loc3=2,500 t, Loc4=4,300 t, Loc5=2,600 t, Loc6=5,500 t.

All other parameters (4-stage tree, 81 scenarios, 2% MIP gap, 60 months) are unchanged.

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'models'))

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df,
    temps_bad_12, temps_normal_12, temps_good_12,
)
from SP import BinaryProgressiveHedging

print('Imports OK')

Imports OK


In [2]:
# 60-cage fleet — use instance.py directly
units_df_60 = units_df.copy().reset_index(drop=True)
loc_mab_60  = loc_mab

print(f'Fleet: {len(units_df_60)} cages, {units_df_60["location"].nunique()} locations')
print(units_df_60.groupby('location').size().to_frame('cages'))

# Individual location MAB sum (the Tangential threshold)
sum_loc_mab = sum(loc_mab_60.values())
print(f'Sum of individual location MABs: {sum_loc_mab/1e6:.1f} kt = {sum_loc_mab/1e3:.0f} t')
print(f'Base regional MAB: {regional_mab/1e3:.0f} t')

Fleet: 60 cages, 6 locations
          cages
location       
Loc1         10
Loc2         12
Loc3          8
Loc4         10
Loc5          8
Loc6         12
Sum of individual location MABs: 24.5 kt = 24500 t
Base regional MAB: 35000 t


In [3]:
# Three regional MAB scenarios (in kg)
MAB_SCENARIOS = [
    ('Relaxed',     35_000_000),   # 35,000 t — base, non-binding
    ('Tangential',  24_000_000),   # 24,000 t — just below sum of location MABs (24,500 t)
    ('Tight',       18_000_000),   # 18,000 t — binding
]

MIP_GAP = 0.02

print('MAB scenarios:')
for name, mab in MAB_SCENARIOS:
    print(f'  {name}: {mab/1e3:.0f} t')

MAB scenarios:
  Relaxed: 35000 t
  Tangential: 24000 t
  Tight: 18000 t


In [ ]:
# Relaxed — 35,000 t (base)
label, reg_mab = MAB_SCENARIOS[0]
print(f'\n{"="*65}')
print(f'  SP — {label} regional MAB: {reg_mab/1e3:.0f} t')
print(f'{"="*65}')

ald = BinaryProgressiveHedging(
    units_df=units_df_60,
    loc_mab=loc_mab_60,
    regional_mab=reg_mab,
    T=T,
    mip_gap=MIP_GAP,
    temps_bad=temps_bad_12,
    temps_normal=temps_normal_12,
    temps_good=temps_good_12,
)
ald.build()
ald.solve()

relaxed_obj   = ald.eval_obj
relaxed_time  = ald.total_time
relaxed_feas  = ald.n_feasible
relaxed_n_sc  = ald.n_scenarios
print(f'  SP = {relaxed_obj/1e6:.1f} MNOK  |  {relaxed_time:.1f}s  |  {relaxed_feas}/{relaxed_n_sc} feasible')
del ald


  SP — Relaxed regional MAB: 35000 t
Set parameter Username
Set parameter LicenseID to value 2786519
Academic license - for non-commercial use only - expires 2027-03-03


In [ ]:
# Tangential — 24,000 t (just below sum of individual location MABs)
label, reg_mab = MAB_SCENARIOS[1]
print(f'\n{"="*65}')
print(f'  SP — {label} regional MAB: {reg_mab/1e3:.0f} t')
print(f'{"="*65}')

ald = BinaryProgressiveHedging(
    units_df=units_df_60,
    loc_mab=loc_mab_60,
    regional_mab=reg_mab,
    T=T,
    mip_gap=MIP_GAP,
    temps_bad=temps_bad_12,
    temps_normal=temps_normal_12,
    temps_good=temps_good_12,
)
ald.build()
ald.solve()

tangential_obj   = ald.eval_obj
tangential_time  = ald.total_time
tangential_feas  = ald.n_feasible
tangential_n_sc  = ald.n_scenarios
print(f'  SP = {tangential_obj/1e6:.1f} MNOK  |  {tangential_time:.1f}s  |  {tangential_feas}/{tangential_n_sc} feasible')
del ald

In [ ]:
# Tight — 18,000 t
label, reg_mab = MAB_SCENARIOS[2]
print(f'\n{"="*65}')
print(f'  SP — {label} regional MAB: {reg_mab/1e3:.0f} t')
print(f'{"="*65}')

ald = BinaryProgressiveHedging(
    units_df=units_df_60,
    loc_mab=loc_mab_60,
    regional_mab=reg_mab,
    T=T,
    mip_gap=MIP_GAP,
    temps_bad=temps_bad_12,
    temps_normal=temps_normal_12,
    temps_good=temps_good_12,
)
ald.build()
ald.solve()

tight_obj   = ald.eval_obj
tight_time  = ald.total_time
tight_feas  = ald.n_feasible
tight_n_sc  = ald.n_scenarios
print(f'  SP = {tight_obj/1e6:.1f} MNOK  |  {tight_time:.1f}s  |  {tight_feas}/{tight_n_sc} feasible')
del ald

In [ ]:
# Results table (Table 11)

rows = [
    {
        'Metric': 'Scenarios',
        'Relaxed (35,000 t)':      f'{relaxed_feas}/{relaxed_n_sc} feasible',
        'Tangentially (24,000 t)': f'{tangential_feas}/{tangential_n_sc} feasible',
        'Tight (18,000 t)':        f'{tight_feas}/{tight_n_sc} feasible',
    },
    {
        'Metric': 'E[obj] (MNOK)',
        'Relaxed (35,000 t)':      f'{relaxed_obj/1e6:.1f}',
        'Tangentially (24,000 t)': f'{tangential_obj/1e6:.1f}',
        'Tight (18,000 t)':        f'{tight_obj/1e6:.1f}',
    },
    {
        'Metric': 'MIP gap tolerance (%)',
        'Relaxed (35,000 t)':      '2',
        'Tangentially (24,000 t)': '2',
        'Tight (18,000 t)':        '2',
    },
    {
        'Metric': 'Wall clock time (s)',
        'Relaxed (35,000 t)':      f'{relaxed_time:.1f}',
        'Tangentially (24,000 t)': f'{tangential_time:.1f}',
        'Tight (18,000 t)':        f'{tight_time:.1f}',
    },
]

df_results = pd.DataFrame(rows).set_index('Metric')
display(df_results)